# 05 — Customer Graph Analysis

Builds the Customer/Email/Phone relationship graph from `ARCHITECTURE.md §9` with NetworkX, computes centrality and connected components, and shows how graph analysis catches duplicate clusters that row-by-row matching (notebook 02) can miss when neither email nor phone alone links two records but a chain of shared attributes does.

Run `make seed` from the repo root before running this notebook. Requires `networkx`.

In [ ]:
from pathlib import Path

import networkx as nx
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
CRM_PATH = ROOT / "data" / "synthetic" / "crm" / "crm_customers.csv"

if not CRM_PATH.exists():
    raise FileNotFoundError("Run `make seed` from the repo root first — see DATA_MODEL.md.")

crm = pd.read_csv(CRM_PATH)
print(f"{len(crm):,} CRM customers")

## Build the graph

Nodes: `Customer`, `Email`, `Phone`, `Address` (city+state as a coarse proxy for a real address node). Edges: `HAS_EMAIL`, `HAS_PHONE`, `LIVES_AT` — a subset of the full node/edge set in `ARCHITECTURE.md §9` (Order/Product/Seller nodes require the Olist join, out of scope for this notebook).

In [ ]:
G = nx.Graph()

for row in crm.itertuples():
    customer_node = ("customer", row.crm_customer_id)
    G.add_node(customer_node, kind="customer", name=row.name)

    if isinstance(row.email, str) and row.email:
        email_node = ("email", row.email.lower())
        G.add_node(email_node, kind="email")
        G.add_edge(customer_node, email_node, relation="HAS_EMAIL")

    phone_digits = "".join(c for c in str(row.phone) if c.isdigit())
    if len(phone_digits) >= 8:
        phone_node = ("phone", phone_digits)
        G.add_node(phone_node, kind="phone")
        G.add_edge(customer_node, phone_node, relation="HAS_PHONE")

    address_node = ("address", f"{row.city}|{row.state}")
    G.add_node(address_node, kind="address")
    G.add_edge(customer_node, address_node, relation="LIVES_AT")

print(f"Graph: {G.number_of_nodes():,} nodes, {G.number_of_edges():,} edges")

## Centrality and structure

High-degree `email`/`phone` nodes are shared by multiple customers — exactly the deterministic-match signal from notebook 02, but here derived from graph structure instead of a groupby.

In [ ]:
degree = dict(G.degree())
shared_contact_nodes = {
    n: d for n, d in degree.items() if n[0] in ("email", "phone") and d > 1
}
print(f"Email/phone nodes shared by >1 customer: {len(shared_contact_nodes):,}")

components = list(nx.connected_components(G))
customer_components = [c for c in components if sum(1 for n in c if n[0] == "customer") > 1]
print(f"Connected components containing >1 customer (excluding address-only links): "
      f"{sum(1 for c in customer_components if any(n[0] in ('email', 'phone') for n in c)):,}")

## Visualize one duplicate cluster

In [ ]:
import matplotlib.pyplot as plt

candidate = next(
    (c for c in customer_components
     if any(n[0] in ("email", "phone") for n in c) and len(c) <= 8),
    None,
)

if candidate:
    subgraph = G.subgraph(candidate)
    color_map = {"customer": "#4C72B0", "email": "#DD8452", "phone": "#55A868", "address": "#C44E52"}
    colors = [color_map[data["kind"]] for _, data in subgraph.nodes(data=True)]
    labels = {n: (G.nodes[n].get("name", n[1]) if n[0] == "customer" else n[1]) for n in subgraph.nodes()}

    plt.figure(figsize=(6, 5))
    pos = nx.spring_layout(subgraph, seed=42)
    nx.draw(subgraph, pos, node_color=colors, with_labels=False, node_size=600, edge_color="#999999")
    nx.draw_networkx_labels(subgraph, pos, labels, font_size=7)
    plt.title("Example duplicate cluster (blue=customer, orange=email, green=phone, red=address)")
    plt.tight_layout()
    plt.show()
else:
    print("No small example cluster found in this run — the dataset's random seed may need a larger --customers count.")

## Maps to the real pipeline

- `graph/networkx/build_graph.py` — the production version of the cell above, extended with `Order`/`Product`/`Seller` nodes from Olist.
- `graph/algorithms/` — betweenness centrality and community detection (e.g. Louvain) on top of what's shown here.
- `graph/queries/duplicate_clusters.md` — the Sprint 6 acceptance-criteria deliverable this notebook is a preview of.
- `02_entity_resolution_golden_record.ipynb` — the row-by-row matcher this graph view cross-checks.